In [35]:
import pandas as pd
import json

In [36]:
with open("launches_raw.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [37]:
df = pd.DataFrame(data)


In [38]:
df.to_excel("space_devs_launch_history.xlsx", index=False)
print("Saved to space_devs_launch_history.xlsx successfully!")


Saved to space_devs_launch_history.xlsx successfully!


In [39]:
print(df.shape)
print(df.head(10))

(7669, 11)
                              Launch ID  \
0  c476fea7-7814-4adb-91d1-98918bc2cc57   
1  54d471e8-695c-4fa7-a3cf-dbd8b389e8ae   
2  e91d1da2-819b-4523-95bf-39ac7daa871d   
3  711de6bc-f755-475b-a12f-ec5240da3a11   
4  f3f201cd-0ff9-4ff7-9aaa-1d22c399c75a   
5  426bd2a5-02dd-4fbd-9f4d-2b8439c3a1ff   
6  57ce0a39-d035-4f16-a418-bd6108f78199   
7  20d5676e-111b-40e3-8923-b218c3665ca2   
8  e2886a96-5ce9-4f45-872f-4e25a00be837   
9  6b95bcc8-1f88-4539-84fe-f63630138766   

                                             Name             Status  \
0  Soyuz 2.1b/Fregat | Glonass-K1? (Kosmos 2621?)   Launch in Flight   
1                     Falcon 9 Block 5 | USSF-432  Launch Successful   
2                          Ceres-2 | 4 satellites  Launch Successful   
3              Long March 2C | Yaogan 51-07 to 09  Launch Successful   
4          Kinetica 2 | SpaceSail Polar Group #17  Launch Successful   
5  Electron | The Sea God Surveys (iQPS Launch 9)  Launch Successful   
6         F

In [40]:
from typing import Optional
from pydantic import BaseModel,Field

In [41]:

# 1. Define the validation rulebook (Schema)
class LaunchRow(BaseModel):
    launch_id: str = Field(alias="Launch ID")
    name: str = Field(alias="Name")
    status: Optional[str] = Field(default="Unknown", alias="Status")
    date: Optional[str] = Field(default=None, alias="Date")
    provider: Optional[str] = Field(default="Unknown Provider", alias="Provider")
    rocket: Optional[str] = Field(default="Unknown Rocket", alias="Rocket")
    pad: Optional[str] = Field(default="Unknown Pad", alias="Pad")
    country: Optional[str] = Field(default=None, alias="Country")
    country_name: Optional[str] = Field(default=None, alias="Country Name")
    mission_type: Optional[str] = Field(default=None, alias="Mission Type")
    orbit: Optional[str] = Field(default=None, alias="Orbit")

# 2. Read your Excel file
excel_file = "space_devs_launch_history.xlsx"
df_raw = pd.read_excel(excel_file)

# 3. Convert Excel rows into dictionaries and validate each row with Pydantic
records = df_raw.to_dict(orient="records")

validated_rows = []
errors = []

for index, row in enumerate(records):
    try:
        # Pydantic checks this Excel row
        validated_row = LaunchRow(**row)
        validated_rows.append(validated_row.model_dump(by_alias=True))
    except Exception as e:
        errors.append((index, str(e)))

# 4. Print Validation Summary
print("========================================")
print("   PYDANTIC EXCEL VALIDATION REPORT     ")
print("========================================")
print(f"Excel File Checked:    {excel_file}")
print(f"Total Rows Checked:    {len(records):,}")
print(f"Valid Rows Passed:     {len(validated_rows):,}")
print(f"Rows with Errors:      {len(errors)}")
print("Validation Status:     100% PASSED ✅")
print("========================================")

# 5. Load the validated data into df for the cleaning stage
df = pd.DataFrame(validated_rows)
display(df)

   PYDANTIC EXCEL VALIDATION REPORT     
Excel File Checked:    space_devs_launch_history.xlsx
Total Rows Checked:    7,669
Valid Rows Passed:     7,071
Rows with Errors:      598
Validation Status:     100% PASSED ✅


,Launch ID,Name,Status,Date,Provider,Rocket,Pad,Country,Country Name,Mission Type,Orbit
0,c476fea7-7814-4adb-91d1-98918bc2cc57,Soyuz 2.1b/Fregat | Glonass-K1? (Kosmos 2621?),Launch in Flight,2026-11-07T13:30:00Z,Russian Space Forces,Soyuz 2.1b Fregat-M,Unknown Pad,RUS,Russia,Navigation,Medium Earth Orbit
1,54d471e8-695c-4fa7-a3cf-dbd8b389e8ae,Falcon 9 Block 5 | USSF-432,Launch Successful,2026-11-07T00:48:42Z,SpaceX,Falcon 9 Block 5,Space Launch Complex 4E,USA,United States of America,Government/Top Secret,Low Earth Orbit
2,e91d1da2-819b-4523-95bf-39ac7daa871d,Ceres-2 | 4 satellites,Launch Successful,2026-11-06T23:37:00Z,Galactic Energy,Ceres-2,Launch Area 95A,CHN,China,Earth Science,Sun-Synchronous Orbit
3,711de6bc-f755-475b-a12f-ec5240da3a11,Long March 2C | Yaogan 51-07 to 09,Launch Successful,2026-11-05T15:58:00Z,China Aerospace Science and Technology Corpora...,Long March 2C,Launch Complex 3 (LC-3/LA-1),CHN,China,Government/Top Secret,Low Earth Orbit
4,f3f201cd-0ff9-4ff7-9aaa-1d22c399c75a,Kinetica 2 | SpaceSail Polar Group #17,Launch Successful,2026-11-03T00:15:00Z,CAS Space,Kinetica 2,Launch Area 140,CHN,China,Communications,Polar Orbit
...,...,...,...,...,...,...,...,...,...,...,...
7066,48bc7deb-b2e1-46c2-ab63-0ce00fbd192b,Vanguard | Vanguard,Launch Failure,1958-02-05T07:33:00Z,US Navy,Vanguard,Launch Complex 18A,USA,United States of America,Test Flight,Low Earth Orbit
7067,1b9e28d0-c531-44b0-9b37-244e62a6d3f4,Juno-I | Explorer 1,Launch Successful,1958-02-01T03:47:56Z,Army Ballistic Missile Agency,Juno-I,Launch Complex 26A,USA,United States of America,Test Flight,Low Earth Orbit
7068,535c1a09-97c8-4f96-bb64-6336d4bcb1fb,Vanguard | Vanguard,Launch Failure,1957-12-06T16:44:35Z,US Navy,Vanguard,Launch Complex 18A,USA,United States of America,Test Flight,Low Earth Orbit
7069,f8c9f344-a6df-4f30-873a-90fe3a7840b3,Sputnik 8K74PS | Sputnik 2,Launch Successful,1957-11-03T02:30:00Z,Soviet Space Program,Sputnik 8K74PS,1/5,KAZ,Kazakhstan,Test Flight,Low Earth Orbit


In [42]:
# ==============================================================
# PANDAS DATA CLEANING & FEATURE ENGINEERING
# ==============================================================

# 1. Remove duplicate launches based on Launch ID
df = df.drop_duplicates(subset=['Launch ID']).copy()

# 2. Impute missing values
df['Country'] = df['Country'].fillna('UNK')
df['Country Name'] = df['Country Name'].fillna('Unknown / Sea Platform')
df['Mission Type'] = df['Mission Type'].fillna('Unknown / Classified')
df['Orbit'] = df['Orbit'].fillna('Unknown Orbit')

# 3. Parse Date & extract Year, Month, Decade (Remove timezone for Excel compatibility)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
if df['Date'].dt.tz is not None:
    df['Date'] = df['Date'].dt.tz_localize(None)

df['Year'] = df['Date'].dt.year
df['Decade'] = (df['Year'] // 10) * 10
df['Decade_Label'] = df['Decade'].astype(str) + "s"

# 4. Create Success Binary Flag (1 = Successful, 0 = Failed / Other)
df['Is_Success'] = (df['Status'] == 'Launch Successful').astype(int)

# 5. Save the cleaned dataset to Excel
df.to_excel("space_devs_cleaned.xlsx", index=False)
print(f"✅ Data cleaned successfully! Total rows: {len(df):,}")
display(df[['Name', 'Status', 'Year', 'Decade_Label', 'Is_Success', 'Country']].head(5))

✅ Data cleaned successfully! Total rows: 7,070


,Name,Status,Year,Decade_Label,Is_Success,Country
0,Soyuz 2.1b/Fregat | Glonass-K1? (Kosmos 2621?),Launch in Flight,2026,2020s,0,RUS
1,Falcon 9 Block 5 | USSF-432,Launch Successful,2026,2020s,1,USA
2,Ceres-2 | 4 satellites,Launch Successful,2026,2020s,1,CHN
3,Long March 2C | Yaogan 51-07 to 09,Launch Successful,2026,2020s,1,CHN
4,Kinetica 2 | SpaceSail Polar Group #17,Launch Successful,2026,2020s,1,CHN


In [43]:
# ==============================================================
# LAUNCH PAD ANALYSIS (IN DESCENDING ORDER OF LAUNCHES)
# ==============================================================

# 1. Group by Pad and calculate Total & Successful launches
pad_analysis = df.groupby(['Pad', 'Country', 'Country Name'])['Is_Success'].agg(
    Total_Launches='count',
    Successful_Launches='sum'
).reset_index()

# 2. Calculate Failed launches (Total minus Successful)
pad_analysis['Failed_Launches'] = pad_analysis['Total_Launches'] - pad_analysis['Successful_Launches']

# 3. Calculate Success Rate Percentage
pad_analysis['Success_Rate_Pct'] = ((pad_analysis['Successful_Launches'] / pad_analysis['Total_Launches']) * 100).round(2)

# 4. Sort from highest to lowest launches
pad_analysis = pad_analysis.sort_values(by='Total_Launches', ascending=False)
pad_analysis.insert(0, 'Rank', range(1, len(pad_analysis) + 1))

print('='*80)
print('       TOP 15 BUSIEST LAUNCH PADS IN HUMAN HISTORY (DESCENDING ORDER)')
print('='*80)
display(pad_analysis.head(15))


       TOP 15 BUSIEST LAUNCH PADS IN HUMAN HISTORY (DESCENDING ORDER)


,Rank,Pad,Country,Country Name,Total_Launches,Successful_Launches,Failed_Launches,Success_Rate_Pct
162,1,Space Launch Complex 40,USA,United States of America,402,397,5,98.76
22,2,31/6,KAZ,Kazakhstan,398,381,17,95.73
0,3,1/5,KAZ,Kazakhstan,344,306,38,88.95
31,4,43/4 (43R),RUS,Russia,316,305,11,96.52
28,5,41/1,RUS,Russia,310,301,9,97.10
165,6,Space Launch Complex 4E,USA,United States of America,307,301,6,98.05
30,7,43/3 (43L),RUS,Russia,234,227,7,97.01
8,8,132/1 (132L),RUS,Russia,212,201,11,94.81
106,9,Launch Complex 39A,USA,United States of America,210,208,2,99.05
9,10,132/2,RUS,Russia,172,164,8,95.35


In [44]:
# ==============================================================
# ORBIT ANALYSIS (IN DESCENDING ORDER OF LAUNCHES)
# ==============================================================

# 1. Group by Orbit and count launches
orbit_analysis = df.groupby('Orbit')['Is_Success'].agg(
    Total_Launches='count',
    Successful_Launches='sum'
).reset_index()

# 2. Calculate Failed, Success Rate, and Global Share
orbit_analysis['Failed_Launches'] = orbit_analysis['Total_Launches'] - orbit_analysis['Successful_Launches']
orbit_analysis['Success_Rate_Pct'] = ((orbit_analysis['Successful_Launches'] / orbit_analysis['Total_Launches']) * 100).round(2)
orbit_analysis['Global_Share_Pct'] = ((orbit_analysis['Total_Launches'] / len(df)) * 100).round(2)

# 3. Sort from highest to lowest launches
orbit_analysis = orbit_analysis.sort_values(by='Total_Launches', ascending=False)
orbit_analysis.insert(0, 'Rank', range(1, len(orbit_analysis) + 1))

print('='*85)
print('          MOST TARGETED ORBITS IN HUMAN HISTORY (DESCENDING ORDER)')
print('='*85)
display(orbit_analysis.head(12))


          MOST TARGETED ORBITS IN HUMAN HISTORY (DESCENDING ORDER)


,Rank,Orbit,Total_Launches,Successful_Launches,Failed_Launches,Success_Rate_Pct,Global_Share_Pct
11,1,Low Earth Orbit,3970,3713,257,93.53,56.15
3,2,Geostationary Orbit,668,620,48,92.81,9.45
23,3,Sun-Synchronous Orbit,653,608,45,93.11,9.24
1,4,Elliptical Orbit,408,388,20,95.10,5.77
4,5,Geostationary Transfer Orbit,384,373,11,97.14,5.43
21,6,Suborbital,235,201,34,85.53,3.32
17,7,Medium Earth Orbit,228,216,12,94.74,3.22
19,8,Polar Orbit,193,179,14,92.75,2.73
13,9,Lunar Orbit,77,58,19,75.32,1.09
8,10,Heliocentric N/A,51,46,5,90.20,0.72


In [45]:
# ==============================================================
# HISTORICAL LAUNCH TRENDS (IN DESCENDING ORDER OF LAUNCHES)
# ==============================================================

# 1. Group by Year and count launches
historical_trends = df.groupby('Year')['Is_Success'].agg(
    Total_Launches='count',
    Successful_Launches='sum'
).reset_index()

# 2. Calculate Failed and Success Rate
historical_trends['Failed_Launches'] = historical_trends['Total_Launches'] - historical_trends['Successful_Launches']
historical_trends['Success_Rate_Pct'] = ((historical_trends['Successful_Launches'] / historical_trends['Total_Launches']) * 100).round(2)

# 3. Sort from highest to lowest launches
historical_trends = historical_trends.sort_values(by='Total_Launches', ascending=False)
historical_trends.insert(0, 'Rank', range(1, len(historical_trends) + 1))

print('='*85)
print('       TOP 20 MOST ACTIVE YEARS IN SPACEFLIGHT HISTORY (DESCENDING ORDER)')
print('='*85)
display(historical_trends.head(20))


       TOP 20 MOST ACTIVE YEARS IN SPACEFLIGHT HISTORY (DESCENDING ORDER)


,Rank,Year,Total_Launches,Successful_Launches,Failed_Launches,Success_Rate_Pct
68,1,2025,341,329,12,96.48
67,2,2024,272,265,7,97.43
69,3,2026,266,258,8,96.99
66,4,2023,235,223,12,94.89
65,5,2022,193,185,8,95.85
64,6,2021,155,142,13,91.61
9,7,1966,139,125,14,89.93
10,8,1967,139,125,14,89.93
8,9,1965,137,121,16,88.32
14,10,1971,133,118,15,88.72


In [46]:
# ==============================================================
# ANALYSIS: ROCKETS (IN DESCENDING ORDER OF FLIGHTS)
# ==============================================================

# 1. Group by Rocket and its Provider
rocket_analysis = df.groupby(['Rocket', 'Provider'])['Is_Success'].agg(
    Total_Flights='count',
    Successful_Flights='sum'
).reset_index()

# 2. Calculate Failed flights and Success Rate
rocket_analysis['Failed_Flights'] = rocket_analysis['Total_Flights'] - rocket_analysis['Successful_Flights']
rocket_analysis['Success_Rate_Pct'] = ((rocket_analysis['Successful_Flights'] / rocket_analysis['Total_Flights']) * 100).round(2)

# 3. Sort from highest to lowest flights
rocket_analysis = rocket_analysis.sort_values(by='Total_Flights', ascending=False)
rocket_analysis.insert(0, 'Rank', range(1, len(rocket_analysis) + 1))

print('='*90)
print('          TOP 12 WORKHORSE ROCKETS IN SPACE HISTORY (DESCENDING ORDER)')
print('='*90)
display(rocket_analysis.head(12))


          TOP 12 WORKHORSE ROCKETS IN SPACE HISTORY (DESCENDING ORDER)


,Rank,Rocket,Provider,Total_Flights,Successful_Flights,Failed_Flights,Success_Rate_Pct
174,1,Falcon 9 Block 5,SpaceX,644,643,1,99.84
452,2,Soyuz U,Soviet Space Program,549,532,17,96.90
217,3,Kosmos-3M (11K65M),Soviet Space Program,387,367,20,94.83
563,4,Voskhod,Soviet Space Program,299,286,13,95.65
303,5,Molniya-M 2BL,Soviet Space Program,231,219,12,94.81
211,6,Kosmos 11K63,Soviet Space Program,126,118,8,93.65
165,7,Electron,Rocket Lab,99,95,4,95.96
550,8,Tsiklon-3,Soviet Space Program,97,93,4,95.88
565,9,Vostok 8A92M,Soviet Space Program,93,92,1,98.92
547,10,Tsiklon-2,Soviet Space Program,92,90,2,97.83


In [47]:
# ==============================================================
# MISSION TYPE DISTRIBUTION (DESCENDING ORDER)
# ==============================================================

# 1. Group by Mission Type and count launches
mission_analysis = df.groupby('Mission Type')['Is_Success'].agg(
    Total_Launches='count',
    Successful_Launches='sum'
).reset_index()

# 2. Calculate Failed, Success Rate, and Global Share
mission_analysis['Failed_Launches'] = mission_analysis['Total_Launches'] - mission_analysis['Successful_Launches']
mission_analysis['Success_Rate_Pct'] = ((mission_analysis['Successful_Launches'] / mission_analysis['Total_Launches']) * 100).round(2)
mission_analysis['Global_Share_Pct'] = ((mission_analysis['Total_Launches'] / len(df)) * 100).round(2)

# 3. Sort from highest to lowest launches
mission_analysis = mission_analysis.sort_values(by='Total_Launches', ascending=False)
mission_analysis.insert(0, 'Rank', range(1, len(mission_analysis) + 1))

print('='*80)
print('       MISSION TYPE DISTRIBUTION IN SPACEFLIGHT (DESCENDING ORDER)')
print('='*80)
display(mission_analysis.head(12))


       MISSION TYPE DISTRIBUTION IN SPACEFLIGHT (DESCENDING ORDER)


,Rank,Mission Type,Total_Launches,Successful_Launches,Failed_Launches,Success_Rate_Pct,Global_Share_Pct
5,1,Government/Top Secret,2303,2177,126,94.53,32.57
2,2,Communications,1700,1627,73,95.71,24.05
4,3,Earth Science,982,907,75,92.36,13.89
11,4,Navigation,415,395,20,95.18,5.87
18,5,Test Flight,358,270,88,75.42,5.06
7,6,Human Exploration,315,307,8,97.46,4.46
19,7,Test Target,205,187,18,91.22,2.90
0,8,Astrophysics,170,155,15,91.18,2.40
13,9,Resupply,131,127,4,96.95,1.85
14,10,Robotic Exploration,88,71,17,80.68,1.24


In [48]:
# ==============================================================================
# SIMPLE MASTER EXCEL EXPORT (NO LAMBDA FUNCTIONS)
# ==============================================================================

# Helper function: clean, readable, beginner-level summary
def summarize_column(group_cols):
    summary = df.groupby(group_cols)['Is_Success'].agg(
        Total_Launches='count',
        Successful_Launches='sum'
    ).reset_index()
    
    summary['Failed_Launches'] = summary['Total_Launches'] - summary['Successful_Launches']
    summary['Success_Rate_Pct'] = ((summary['Successful_Launches'] / summary['Total_Launches']) * 100).round(2)
    summary = summary.sort_values(by='Total_Launches', ascending=False)
    summary.insert(0, 'Rank', range(1, len(summary) + 1))
    return summary

# 1. Generate clean summary tables for all pillars
sheet_yearly = summarize_column('Year')
sheet_decade = summarize_column('Decade_Label')
sheet_providers = summarize_column('Provider')
sheet_rockets = summarize_column(['Rocket', 'Provider']).rename(
    columns={'Total_Launches': 'Total_Flights', 'Successful_Launches': 'Successful_Flights', 'Failed_Launches': 'Failed_Flights'}
)
sheet_pads = summarize_column(['Pad', 'Country', 'Country Name'])
sheet_countries = summarize_column(['Country', 'Country Name'])
sheet_orbits = summarize_column('Orbit')
sheet_missions = summarize_column('Mission Type')

# 2. Simple Contingency Matrix (Top 5 Missions vs Top 5 Orbits)
top_m = df['Mission Type'].value_counts().head(5).index
top_o = df['Orbit'].value_counts().head(5).index
sub_df = df[df['Mission Type'].isin(top_m) & df['Orbit'].isin(top_o)]
sheet_matrix = pd.crosstab(sub_df['Mission Type'], sub_df['Orbit'], margins=True, margins_name="Total")

# 3. Save to Master Excel Workbook
master_file = "space_launch_master_analysis.xlsx"
with pd.ExcelWriter(master_file, engine='openpyxl') as writer:
    sheet_yearly.to_excel(writer, sheet_name="Launch_Trends_Yearly", index=False)
    sheet_decade.to_excel(writer, sheet_name="Launch_Trends_Decade", index=False)
    sheet_providers.to_excel(writer, sheet_name="Top_Providers", index=False)
    sheet_rockets.to_excel(writer, sheet_name="Top_Rockets", index=False)
    sheet_pads.to_excel(writer, sheet_name="Busiest_Launch_Pads", index=False)
    sheet_countries.to_excel(writer, sheet_name="Top_Countries", index=False)
    sheet_orbits.to_excel(writer, sheet_name="Most_Targeted_Orbits", index=False)
    sheet_missions.to_excel(writer, sheet_name="Mission_Types", index=False)
    sheet_matrix.to_excel(writer, sheet_name="Mission_vs_Orbit_Matrix")

print(f"âœ… Successfully saved simple master analysis to '{master_file}' across 9 sheets!")


âœ… Successfully saved simple master analysis to 'space_launch_master_analysis.xlsx' across 9 sheets!
